In [30]:
%load_ext autoreload
%autoreload 1
%aimport classes.GaloisField
%aimport classes.GolayDecoder

import numpy as np

from classes.GaloisField import *
from classes.GaloisPoly  import *
from classes.GolayEncoder import GolayEncoder
from classes.GolayDecoder import GolayDecoder

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


## Generate Galois Field

In [31]:
gf              = GaloisField(1,0b11)
encoder_model   = GolayEncoder()
k               = encoder_model._k
n               = encoder_model._n


Field Closed Succesfully!, 1 Non-Zero Elements


## 1. Codewords Test

### Generate all codewords

In [32]:
encoder_output = None
for w in range(2**k):
    w   = gf.do_unpack(w, bit_width=k)
    cw  = encoder_model.encode(w)
    encoder_output = cw if encoder_output is None else np.vstack((encoder_output, cw))

n_codewords = len(encoder_output)

### Decoding of codewords

In [33]:
n_codewords
#encoder_output 
decoder_model = GolayDecoder()

decoder_output = []

for i in range(n_codewords):
    decoder_output.append(decoder_model.correct(encoder_output[i]))

#print(decoder_output)

Field Closed Succesfully!, 1 Non-Zero Elements


### Error (`o_err`)

In [34]:
o_err           = []
corrected       = []
uncorrectable   = []

o_corrected     = []
o_uncorrectable = []

for i in range(n_codewords):
    # 0: i_rx, 1: o_corrected, 2: o_uncorrectable
    o_err.append(decoder_output[i][0] - encoder_output[i])
    # Flags for uvm
    o_corrected.append(int(decoder_output[i][1]))
    o_uncorrectable.append(int(decoder_output[i][2]))
    # Corrected and uncorrectable bool
    corrected.append(decoder_output[i][1])
    uncorrectable.append(decoder_output[i][2])

# print(o_err, o_corrected, o_uncorrectable)

print(np.array(o_err).shape)
print(np.array(o_corrected).shape)
print(np.array(o_uncorrectable).shape)



(4096, 24)
(4096,)
(4096,)


In [35]:
def write_sv_array(buf, name, width_label, width_bits, values, comment=None):
    """Write a plain SV array decl: bit [width_label] name [len(values)] = '{ ... };
    `width_label` is the bracket text (e.g. "NB_CODEWORD-1:0"), or None for a 1-bit `bit` array."""
    if comment:
        buf.write(f"\t// {comment}\n")
    decl = f"bit {name}" if width_label is None else f"bit [{width_label}] {name}"
    buf.write(f"\t{decl} [{len(values)}] = '{{\n")
    for i, v in enumerate(values):
        literal = f"1'b{int(v)}" if width_label is None else f"{width_bits}'b{v:0{width_bits}b}"
        sep = "};\n" if i == len(values) - 1 else ",\n"
        buf.write(f"\t\t{literal}\t{sep}")
    buf.write("\n")

In [36]:
from io import StringIO

filenames = [
    "outputs/decoder_testing/codewords/codewords_testing_golay_code.svh",
    "../implem/golay_decoder/decoder/decoder.uvm/sequences/decoder_testing/codewords/codewords_testing_golay_code.svh"
]

file_content = StringIO()

# Received codewords (no errors)
write_sv_array(file_content, "golay_code", "NB_CODEWORD-1:0", 24,
               [gf.do_pack(cw) for cw in encoder_output],
               comment="Received codewords (no errors)")

# Decoded codewords
write_sv_array(file_content, "decoded_data", "NB_CODEWORD-1:0", 24,
               [gf.do_pack(cw[0]) for cw in decoder_output],
               comment="Decoded/corrected codewords")

# Errors
write_sv_array(file_content, "err", "NB_CODEWORD-1:0", 24,
               [gf.do_pack(e) for e in o_err],
               comment="Error pattern (rx XOR original codeword)")

# Corrected received words flag
write_sv_array(file_content, "corrected_data", None, 1, o_corrected,
               comment="Corrected flag")

# Uncorrectable received words flag
write_sv_array(file_content, "uncorrectable", None, 1, o_uncorrectable,
               comment="Uncorrectable flag")

# Write the same content to both files
content = file_content.getvalue()

for filename in filenames:

    with open(filename, "w") as file:
        file.write(content)

## 2. Codewords with errors Test

In [37]:
encoder_output 

received_msg_with_error = []

n_errors = 5 # 4 errors limit

for n in range(1, n_errors):
    codewords_n_errors = []
    # Select random n positions
    error_positions = np.random.choice(24, n, replace=False)

    for i in range(n_codewords):
        codeword = encoder_output[i].copy()

        # Put errores
        for pos in error_positions:
            codeword[pos] ^= 1

        # Guardar la codeword completa
        codewords_n_errors.append(codeword)

    received_msg_with_error.append(codewords_n_errors)

#print(received_msg_with_error[0][1][3])

# Dimensions

# received_msg_with_error[0] → codewords with 1 error
# received_msg_with_error[1] → codewords with 2 errors
# received_msg_with_error[2] → codewords with 3 errors
# received_msg_with_error[3] → codewords with 4 errors

# received_msg_with_error[0][0] → first codeword with 1 error
# received_msg_with_error[0][1] → second codeword with 1 error
# codeword with 24-bits

# received_msg_with_error[0][1][0] → first bit of codeword with 1 error

# received_msg_with_error[error_group][codeword][bit]
#                          │            │          │
#                          │            │          └── 0 ... 23
#                          │            │
#                          │            └──────────── 0 ... n_codewords-1
#                          │
#                          └───────────────────────── 0 ... 3


### Decoding of codewords with errors

In [38]:
o_no_cw_err = []
o_no_cw_msg = []
o_no_cw_uncorrectable = []
o_no_cw_corrected = []

for n in range(n_errors - 1):

    # Resultados para esta cantidad de errores
    err_n = []
    msg_n = []
    uncorrectable_n = []
    corrected_n = []

    for i in range(n_codewords):
        received = received_msg_with_error[n][i]
        # Decoder
        decoded = decoder_model.correct(received)
        # Syndrome
        s, q = decoder_model.get_s_q(received)
        # Error pattern
        if decoder_model._gf.do_pack(s) != 0:
            err_n.append(decoder_model.get_error(s, q))
        else:
            err_n.append(0)
        # Message/corrected codeword
        msg_n.append(decoded[0])

        # Flags
        uncorrectable_n.append(int(decoded[1]))
        corrected_n.append(int(decoded[2]))

    # Guardar resultados de este número de errores
    o_no_cw_err.append(err_n)
    o_no_cw_msg.append(msg_n)
    o_no_cw_uncorrectable.append(uncorrectable_n)
    o_no_cw_corrected.append(corrected_n)

### Create files `received words with n-errors`

In [39]:
for n in range(n_errors - 1):

    filenames = [
        f"outputs/decoder_testing/error_{n+1}_vectors/codewords_testing_golay_{n+1}_error.svh",
        f"../implem/golay_decoder/decoder/decoder.uvm/sequences/decoder_testing/error_{n+1}_vectors/codewords_testing_golay_{n+1}_error.svh"
    ]

    file_content = StringIO()

    # Received codewords with n+1 induced bit errors
    write_sv_array(file_content, f"rx_{n+1}_error_vector", "NB_CODEWORD-1:0", 24,
                   [gf.do_pack(cw) for cw in received_msg_with_error[n]],
                   comment=f"Received codewords with {n+1} induced bit error(s)")

    # Decoded codewords
    write_sv_array(file_content, f"msg_{n+1}_error_vector", "NB_CODEWORD-1:0", 24,
                   [gf.do_pack(cw) for cw in o_no_cw_msg[n]],
                   comment=f"Decoded/corrected codewords ({n+1}-bit error input)")

    # Errors (recovered error pattern, zero vector when the syndrome is 0)
    write_sv_array(file_content, f"err_pattern_{n+1}_error", "NB_CODEWORD-1:0", 24,
                   [gf.do_pack(err if err is not None else [0] * 24) for err in o_no_cw_err[n]],
                   comment=f"Recovered error pattern ({n+1}-bit error input)")

    # Corrected received words flag
    write_sv_array(file_content, f"corrected_flag_{n+1}_error", None, 1, o_no_cw_corrected[n],
                   comment=f"Corrected flag ({n+1}-bit error input)")

    # Uncorrectable received words flag
    write_sv_array(file_content, f"uncorrectable_flag_{n+1}_error", None, 1, o_no_cw_uncorrectable[n],
                   comment=f"Uncorrectable flag ({n+1}-bit error input)")

    content = file_content.getvalue()

    for filename in filenames:

        with open(filename, "w") as file:
            file.write(content)

## golay (24,12) decoding example

In [40]:
r = encoder_output[3576]
# Generate random error for r (word received/transmitted)
r = r ^ np.array([
    [1,0,0,1,0,0,0,1,0,0,0,0]   ,
    [0,0,0,0,0,0,0,0,0,0,0,0]   ]).flatten()

decoder_model = GolayDecoder()

w, corrected, uncorrectable = decoder_model.decode(r)

# r: word with errors
# word decoded (possible codeword),
# flags (corrected, uncorrectable)
# encoder output (word received transmitted)
r, decoder_model.decode(r), encoder_output[3576]

Field Closed Succesfully!, 1 Non-Zero Elements


(array([0, 1, 0, 0, 1, 1, 1, 0, 1, 0, 0, 0, 0, 0, 1, 1, 0, 1, 1, 0, 1, 1,
        1, 1]),
 (array([1, 1, 0, 1, 1, 1, 1, 1, 1, 0, 0, 0]), True, False),
 array([1, 1, 0, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 1, 1, 0, 1, 1, 0, 1, 1,
        1, 1], dtype=uint8))

In [41]:
# decode all codewords, no errors
for cw in encoder_output:
    w, corrected, uncorrectable = decoder_model.decode(cw, full_codeword=True)
    assert(np.all(w == cw))
    assert(not corrected and not uncorrectable)

for cw in encoder_output:
    error_seed      = np.random.randint(0, 0b11111)
    # decimal error_seed converted into n-bits
    error           = decoder_model._gf.do_unpack(error_seed, bit_width=decoder_model._n)
    # calculate hamming weight
    error_weight    = decoder_model._gf.hamming_weight(error)
    
    np.random.shuffle(error)
    w, corrected, uncorrectable = decoder_model.decode(cw ^ error, full_codeword=True)
    
    # no errors
    if error_weight == 0:
        assert(np.all(w == cw))
        assert(not corrected and not uncorrectable)
    # 1 to 3 errors
    elif 1 <= error_weight <= 3:
        assert(np.all(w == cw))
        assert(corrected and not uncorrectable)
    # 4 errors
    else:
        assert(not np.all(w == cw))
        assert(not corrected and uncorrectable)

    # five or more errors this decoding fails, recovered bits and flags are invalid


## Decoding test with selected values 

Verify the following values:

$$ r_{1} = 0xA5D9A6 $$
$$ r_{2} = 0xA5F9A4 $$
$$ r_{3} = 0xA5C9AA $$

### 1. Obtain values corrected.

In [42]:
import numpy as np

rx_test = [0xA5D9A6, 0xA5F9A4, 0xA5C9AA]

rx_test_array = [
    np.array([int(bit) for bit in format(x, '024b')], dtype=np.uint8)
    for x in rx_test]

rx_test_array

decoded_rx_test        = []
decoded_rx_test_binary = []

for i in range(len(rx_test_array)):
    decoded_rx_test.append(decoder_model.decode(rx_test_array[i], False))

# print(decoded_rx_test)

for decoded, valid, uncorrectable in decoded_rx_test:
    decoded_rx_test_binary.append(
        (decoded, int(valid), int(uncorrectable))
    )

# decoded word, o_corrected, o_uncorrected
decoded_rx_test_binary

[(array([1, 0, 1, 0, 0, 1, 0, 1, 1, 1, 0, 0], dtype=uint8), 1, 0),
 (array([1, 0, 1, 0, 0, 1, 0, 1, 1, 1, 0, 0], dtype=uint8), 1, 0),
 (array([1, 0, 1, 0, 0, 1, 0, 1, 1, 1, 0, 0], dtype=uint8), 0, 1)]

### 2. Obtain mask error

In [43]:
s_q_vectors  = []
error_vector = []

for i in range (len(rx_test_array)):
    s_q_vectors.append(decoder_model.get_s_q(rx_test_array[i]))
    error_vector.append(decoder_model.get_error(s_q_vectors[i][0], s_q_vectors[i][1]))
    if error_vector[i] is None:
        error_vector[i] = [0]*24
    
error_vector

[array([0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        1, 1], dtype=uint8),
 array([0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 1], dtype=uint8),
 [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]]

In [44]:
from io import StringIO

filenames = [
    "outputs/decoding_vectors_test/codewords_testing_golay_test.svh",
    "../implem/golay_decoder/decoder/decoder.uvm/sequences/decoding_vectors_test/codewords_testing_golay_test.svh"
]
# /home/abel/Desktop/desktop/github_repos/fec-lab2/implem/golay_decoder/decoder/decoder.uvm/sequences
file_content = StringIO()

# Received test vectors
write_sv_array(file_content, "rx_test_vectors", "NB_CODEWORD-1:0", 24,
               [int("".join(map(str, rx)), 2) for rx in rx_test_array],
               comment="Selected received test vectors: 0xA5D9A6, 0xA5F9A4, 0xA5C9AA")

# Decoded message
write_sv_array(file_content, "msg_test_vectors", "NB_WORD-1:0", 12,
               [int("".join(map(str, d[0])), 2) for d in decoded_rx_test_binary],
               comment="Decoded message for the selected test vectors")

# Error vectors
write_sv_array(file_content, "err_pattern_test_vectors", "NB_CODEWORD-1:0", 24,
               [int("".join(map(str, err)), 2) for err in error_vector],
               comment="Recovered error pattern for the selected test vectors")

# Corrected flag
write_sv_array(file_content, "corrected_flag_test_vectors", None, 1,
               [d[1] for d in decoded_rx_test_binary],
               comment="Corrected flag for the selected test vectors")

# Uncorrectable flag
write_sv_array(file_content, "uncorrectable_flag_test_vectors", None, 1,
               [d[2] for d in decoded_rx_test_binary],
               comment="Uncorrectable flag for the selected test vectors")

# Write the same content to both files
content = file_content.getvalue()

for filename in filenames:
    with open(filename, "w") as file:
        file.write(content)